# Storing Embeddings in Delta Tables

## Selecting a Small Sample for Real Embedding Generation

In [0]:
%python
sample_df = spark.table("pipeline_catalog.tabular_pipeline.customer_features_final") \
    .select("customer_id", "review_text") \
    .limit(10)

sample_df.createOrReplaceTempView("review_sample")
sample_df.show(truncate=False)

## Generating Real Embeddings Using ai_query

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW review_sample_with_embeddings AS
SELECT
    customer_id,
    review_text,
    ai_query('databricks-gte-large-en', review_text) AS embedding
FROM review_sample;

SELECT customer_id, review_text, size(embedding) AS embedding_length
FROM review_sample_with_embeddings;

## Adding Model Metadata Before Writing

In [0]:
%python
from pyspark.sql.functions import lit, size, current_timestamp

embeddings_df = spark.table("review_sample_with_embeddings") \
    .withColumn("embedding_model", lit("databricks-gte-large-en")) \
    .withColumn("embedding_dimension", size("embedding")) \
    .withColumn("generated_at", current_timestamp())

embeddings_df.select("customer_id", "embedding_model", "embedding_dimension", "generated_at").show()

## Writing to the Embedding Feature Table

In [0]:
%python
embeddings_df.select(
    "customer_id", "review_text", "embedding",
    "embedding_model", "embedding_dimension", "generated_at"
).write.format("delta").mode("append").saveAsTable(
    "pipeline_catalog.tabular_pipeline.customer_review_embeddings"
)

print("Embeddings written successfully")

## Verifying the Stored Embeddings

In [0]:
%sql
SELECT customer_id, embedding_model, embedding_dimension, generated_at,
       size(embedding) AS actual_array_length
FROM pipeline_catalog.tabular_pipeline.customer_review_embeddings;

## Proving the Dimension Constraint Actually Works

In [0]:
%sql
INSERT INTO pipeline_catalog.tabular_pipeline.customer_review_embeddings
VALUES (
    9999,
    'Fake test review',
    array(0.1, 0.2, 0.3),
    'databricks-gte-large-en',
    3,
    current_timestamp()
);